# ResBlock3D

Now that we understand how `CausalConv3D` works, we can start building larger components used inside modern video VAEs.

One of the most important building blocks is the:

```text
ResBlock3D
```

---

Since we are working on a video, we have 2 diffferent things to work on:

- Spatial information
- Temporal information
always seperate these two things and think how the block is processing both of them 

Now let’s first look at the structure of the block so you can better visualize how the input flows through it:

```text
x → GroupNorm → SiLU → CausalConv3D
  → GroupNorm → SiLU → CausalConv3D
  → + skip(x)
```

I’ll explain all the parts one by one, but before that, let’s first understand what this block as a whole is doing and why we need it.

---

Thinking on the Spatial side, if you remember `CausalConv3D` , for the spatial side, it behaves very similarly to a normal convolution layer.
So here, the reblock3d is simple extracting features from the input.
In the early layers, it learns simple things like:

- edges
- textures
- corners

Then as we go deeper into the network, it starts learning more complex structures like:

- shapes
- objects
- scene layouts

This is very similar to how standard CNNs process images.

---

Thinking on the temporal side, In `CausalConv3D`, we usually use a temporal kernel size of `3` or `5`. So what this block is doing is, its looking at those 3 frames at a time and understanding the "physics" of the scene. 
For Example: If you are halfway through a blink in Frame 1 and 2, the 3D kernel helps the model realize that in Frame 3, the eye should be closing further, not suddenly popping wide open.

If you say 3 or 5 frames are very less, we dont have only 1 resblock3d, we have multiple stacked over each other so

```text
Layer 1 sees 3 frames
Layer 2 sees 3 "Layer 1 summaries"
Layer 3 sees even larger summaries
```
By the time you get to the deep layers, the **Receptive Field** has grown to cover 15 or 20 frames.

So long story short >  Looking at 3 frames allows the model to calculate **change**. Once it knows the "change" (the delta), it can ensure that the motion is mathematically continuous rather than a series of disconnected snapshots.


Now other things are there to stablity the gradint flow and it introduce nonlinerality same as done LLMs
Its called 'Residual Block' because its use skip conenction.

We use prenorm instead of postnorm to keep the backward gradient path cleaner and more stable.

We used GroupNorm here instead of BatchNorm or LayerNorm becasue:

- BatchNorm depends on batch statistics, but in video generation:

- batch sizes are often very small
- videos consume huge GPU memory
- batch statistics become unstable

So BatchNorm performs poorly.

---

For code and more understanding such as whats groupnorm and why not use layernorm or batch norm here, see the notebook [githublink]


# ✅ Why GroupNorm Works Better

GroupNorm splits channels into groups and normalizes them independently.

For example:

```text
128 channels
↓
32 groups
↓
4 channels per group
```

This makes normalization:

- independent of batch size
- more stable for diffusion training
- better for video models

---

# 🛠️ Full ResBlock3D Implementation

```python
class ResBlock3D(nn.Module):
    """
    3D residual block for the VAE encoder/decoder.

    Structure:
        x → Norm → SiLU → Conv3D(3×3×3)
          → Norm → SiLU → Conv3D(3×3×3)
          → + skip(x)

    The skip connection uses a 1×1×1 conv if in/out channels differ,
    otherwise it's an identity.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        num_groups: int = 32,
    ):
        super().__init__()

        self.norm1 = get_norm(in_channels, num_groups)
        self.conv1 = CausalConv3d(in_channels, out_channels, kernel_size=3)

        self.norm2 = get_norm(out_channels, num_groups)
        self.conv2 = CausalConv3d(out_channels, out_channels, kernel_size=3)

        self.act = nn.SiLU()

        if in_channels != out_channels:
            self.skip = CausalConv3d_1x1(in_channels, out_channels)
        else:
            self.skip = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        residual = self.skip(x)

        x = self.act(self.norm1(x))
        x = self.conv1(x)

        x = self.act(self.norm2(x))
        x = self.conv2(x)

        return x + residual
```
